# APOStrategyBaseJ

> JAX base class for partially observable CRLD agents in strategy space.

In [ ]:
#| default_exp Agents/POStrategyBaseJ

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import jax
import numpy as np

import jax.numpy as jnp
from jax import jit
from functools import partial

from fastcore.utils import *

from pyCRLD.Agents.BaseJ import abaseJ
from pyCRLD.Agents.POBaseJ import aPObaseJ
from pyCRLD.Agents.StrategyBaseJ import strategybaseJ
from pyCRLD.Utils.HelpersJ import *

In [ ]:
#| export
class POstrategybaseJ(aPObaseJ, strategybaseJ):
    """JAX base class for partially observable TD reinforcement learning in policy space."""

    def __init__(self, env, learning_rates, discount_factors,
                 choice_intensities=1, **kwargs):
        self.env = env
        Tt = env.T
        assert np.allclose(Tt.sum(-1), 1)
        Rt = env.R
        Ot = env.O
        super().__init__(Tt, Rt, Ot, discount_factors, **kwargs)
        assert np.allclose(env.F, 0), 'PO learning with final states not defined.'

        self.alpha = make_variable_vectorJ(learning_rates, self.N)
        self.beta = make_variable_vectorJ(choice_intensities, self.N)
        self.TDerror = self.RPEioa

In [ ]:
#| export
@patch
def random_softmax_policy(self: POstrategybaseJ,
                           key: jnp.ndarray = None  # JAX PRNG key
                           ) -> jnp.ndarray:
    """Softmax policy with random probabilities. Uses jax.random if key provided."""
    if key is not None:
        logits = jax.random.normal(key, shape=(self.N, self.Q, self.M))
    else:
        logits = jnp.array(np.random.randn(self.N, self.Q, self.M))
    expQ = jnp.exp(logits)
    return expQ / expQ.sum(axis=-1, keepdims=True)

@patch
def zero_intelligence_policy(self: POstrategybaseJ) -> jnp.ndarray:
    """Policy with equal probabilities."""
    return jnp.ones((self.N, self.Q, self.M)) / float(self.M)

## Tests

In [ ]:
# Test with UncertainSocialDilemma (has partial observability) or just check init
from pyCRLD.Environments.SocialDilemmaJ import SocialDilemmaJ

env = SocialDilemmaJ(R=3., T=5., S=0., P=1.)

class _TestPOStrat(POstrategybaseJ):
    @partial(jit, static_argnums=0)
    def RPEioa(self, X, norm=False):
        return jnp.zeros_like(X)

agent = _TestPOStrat(env, learning_rates=0.1, discount_factors=0.9)
X0 = agent.zero_intelligence_policy()
assert X0.shape == (2, 1, 2)  # N, Q, M
assert jnp.allclose(X0.sum(-1), 1.0)

In [ ]:
key = jax.random.PRNGKey(7)
Xr = agent.random_softmax_policy(key=key)
assert Xr.shape == (2, 1, 2)
assert jnp.allclose(Xr.sum(-1), 1.0, atol=1e-5)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()